# Phase 3 Checks: Team Strength and Goalie Ratings

Verifies `src/processing/team_strength.py`, `src/processing/goalie.py` and
`src/models/stochastic/goalie.py`. Re-run after any change to those files or to `shot_xg`.

**Sections**
1. Setup
2. Team-game table: counts and totals
3. Rolling estimates: no leakage from the future
4. Do team estimates predict the next game? *(the most important team check)*
5. Goalie table: counts and totals
6. Goalie ratings: sanity
7. Do goalie ratings predict the next season? *(the most important goalie check, and how to choose `k`)*

Each check prints **PASS** or **CHECK**. A CHECK isn't always a bug; read the note next to it.

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys; sys.path.insert(0, "..")
import numpy as np
import pandas as pd

from src.data.db import get_connection
from src.processing.team_strength import team_game_xg, rolling_strength, strength_as_of
from src.processing.goalie import goalie_game_xg
from src.models.stochastic.goalie import goalie_rating_as_of

def check(label, ok, note=""):
    print(f"{'PASS ' if ok else 'CHECK'}  {label}" + (f"   ({note})" if note else ""))

conn = get_connection()
tg = team_game_xg(conn)
rs = rolling_strength(tg)
gg = goalie_game_xg(conn)
print(f"team-games: {len(tg):,}   goalie-games: {len(gg):,}")

team-games: 13,120   goalie-games: 13,919


## 2. Team-game table: counts and totals

In [2]:
n_games = tg.game_id.nunique()
shot_xg_season = pd.read_sql("SELECT season, SUM(xg) AS xg FROM shot_xg GROUP BY season", conn).set_index("season").xg

check("two rows per game", len(tg) == 2 * n_games, f"{len(tg):,} rows, {n_games:,} games")
check("every game has one home and one away row", (tg.groupby("game_id").is_home.sum() == 1).all())
check("xGF per season matches shot_xg",
      np.allclose(tg.groupby("season").xgf.sum().sort_index(), shot_xg_season.sort_index()))
check("league xGF = league xGA", abs(tg.xgf.sum() - tg.xga.sum()) < 1e-6)
check("game length between 60 and 65 minutes", tg.minutes.between(60, 65.01).all(),
      f"min {tg.minutes.min():.1f}, max {tg.minutes.max():.1f}")
check("no missing values", not tg[["xgf", "xga", "gf", "ga", "minutes"]].isna().any().any())

PASS   two rows per game   (13,120 rows, 6,560 games)
PASS   every game has one home and one away row
PASS   xGF per season matches shot_xg
PASS   league xGF = league xGA
PASS   game length between 60 and 65 minutes   (min 60.0, max 65.0)
PASS   no missing values


## 3. Rolling estimates: no leakage from the future

Recompute one team's pre-game estimate by hand from its previous 25 games and compare. If the
game itself leaked into its own estimate, the numbers would differ.

In [3]:
row = rs[rs.games_in_window == 25].iloc[len(rs[rs.games_in_window == 25]) // 2]
team_games = tg[tg.team == row.team].sort_values(["game_date", "game_id"]).reset_index(drop=True)
pos = team_games.index[team_games.game_id == row.game_id][0]
prev = team_games.iloc[pos - 25:pos]

by_hand = prev.xgf.sum() / prev.minutes.sum() * 60
leaky = team_games.iloc[pos - 24:pos + 1]
leaky_value = leaky.xgf.sum() / leaky.minutes.sum() * 60

print(f"{row.team} going into game {row.game_id} ({row.game_date})")
print(f"  rolling_strength: {row.xgf60:.4f}   by hand: {by_hand:.4f}   if the game leaked in: {leaky_value:.4f}")
check("pre-game estimate uses only earlier games", abs(row.xgf60 - by_hand) < 1e-9)

asof = strength_as_of(tg, row.game_date).set_index("team").loc[row.team]
check("strength_as_of matches rolling_strength", abs(asof.xgf60 - row.xgf60) < 1e-9)

NJD going into game 2023020206 (2023-11-10)
  rolling_strength: 3.2212   by hand: 3.2212   if the game leaked in: 3.1664
PASS   pre-game estimate uses only earlier games
PASS   strength_as_of matches rolling_strength


## 4. Do team estimates predict the next game?

For each game, combine both teams' pre-game estimates into an expected edge for the home team:

> edge = (home xGF/60 − home xGA/60) − (away xGF/60 − away xGA/60)

and compare it with what happened (home goals − away goals). Two comparisons:

- **xG vs goals.** The same rolling estimate built from actual goals instead of xG. xG should
  predict better: goals are noisier, so a team's recent goal record says less about its next game.
- **Window size.** 10, 25 and 50 games. Too short is noisy; too long misses changes in a team.

Only games where both teams have at least 10 games of history count.

In [4]:
def edge_vs_result(strength, min_games=10):
    s = strength[["game_id", "is_home", "xgf60", "xga60", "games_in_window", "gf", "ga"]]
    home = s[s.is_home == 1].set_index("game_id")
    away = s[s.is_home == 0].set_index("game_id")
    both = home.join(away, lsuffix="_h", rsuffix="_a")
    both = both[(both.games_in_window_h >= min_games) & (both.games_in_window_a >= min_games)]
    edge = (both.xgf60_h - both.xga60_h) - (both.xgf60_a - both.xga60_a)
    result = both.gf_h - both.ga_h
    decided = result != 0
    return {
        "games": len(both),
        "corr_with_goal_diff": edge.corr(result),
        "picks_winner": (np.sign(edge[decided]) == np.sign(result[decided])).mean(),
    }

rows = []
for window in [10, 25, 50]:
    xg_based = rolling_strength(tg, window=window)
    goal_based = rolling_strength(tg.assign(xgf=tg.gf, xga=tg.ga), window=window)
    for name, strength in [("xG", xg_based), ("goals", goal_based)]:
        rows.append({"window": window, "built_from": name, **edge_vs_result(strength)})

results = pd.DataFrame(rows)
print(results.round(3).to_string(index=False))

w25 = results[results.window == 25].set_index("built_from")
print()
check("xG-based edge predicts better than goals-based",
      w25.loc["xG", "corr_with_goal_diff"] > w25.loc["goals", "corr_with_goal_diff"])
check("edge picks the winner more than half the time", w25.loc["xG", "picks_winner"] > 0.5,
      "NHL games are close to coin flips; 55-60% is a good result")

 window built_from  games  corr_with_goal_diff  picks_winner
     10         xG   6381                0.233         0.582
     10      goals   6381                0.168         0.567
     25         xG   6381                0.251         0.593
     25      goals   6381                0.206         0.573
     50         xG   6381                0.251         0.591
     50      goals   6381                0.240         0.592

PASS   xG-based edge predicts better than goals-based
PASS   edge picks the winner more than half the time   (NHL games are close to coin flips; 55-60% is a good result)


**Reading section 4.** Single games are very random in hockey, so even a good estimate
correlates only weakly with one game's goal differential (around 0.15–0.25 is typical) and picks
the winner roughly 55–60% of the time. What matters is the comparisons: xG should beat goals, and
the best window is the one with the highest correlation. If 50 clearly beats 25, consider changing
the default in `rolling_strength`.

## 5. Goalie table: counts and totals

In [5]:
goals_season = pd.read_sql("""
    SELECT x.season, SUM(x.xg) AS xg, SUM(CASE WHEN se.event_type = 'goal' THEN 1 ELSE 0 END) AS goals
    FROM shot_xg x JOIN shot_events se ON se.game_id = x.game_id AND se.event_id = x.event_id
    GROUP BY x.season""", conn).set_index("season")
by_season = gg.groupby("season")[["xga", "ga", "gsax"]].sum()
print(by_season.round(1))
print()
check("goalie xGA per season matches shot_xg", np.allclose(by_season.xga, goals_season.xg),
      "a gap means shots with no goalie_id")
check("goalie GA per season matches goals in shot_xg", (by_season.ga == goals_season.goals).all())
check("every goalie-game has a name", gg.goalie_name.notna().all(),
      f"{gg.goalie_name.isna().mean():.1%} missing")
games_per = gg.groupby("game_id").size()
check("one or two goalies per team per game", games_per.between(2, 4).all(),
      f"{(games_per > 2).mean():.1%} of games had a goalie change")

             xga    ga   gsax
season                       
20212022  7922.2  7674  248.2
20222023  8051.3  7810  241.3
20232024  7438.8  7639 -200.2
20242025  7150.7  7377 -226.3
20252026  7759.4  7578  181.4

PASS   goalie xGA per season matches shot_xg   (a gap means shots with no goalie_id)
PASS   goalie GA per season matches goals in shot_xg
PASS   every goalie-game has a name   (0.0% missing)
PASS   one or two goalies per team per game   (11.8% of games had a goalie change)


## 6. Goalie ratings: sanity

In [6]:
last_date = pd.to_datetime(gg.game_date).max() + pd.Timedelta(days=1)
ratings = goalie_rating_as_of(gg, last_date)
cols = ["goalie_name", "team", "games", "xga", "ga", "gsax", "theta_mean", "theta_lo", "theta_hi"]

big = ratings[ratings.games >= ratings.games.quantile(0.75)]
small = ratings[ratings.games <= 5]
print("Best (big samples):");  print(big.head(5)[cols].round(3).to_string(index=False))
print("\nWorst (big samples):"); print(big.tail(5)[cols].round(3).to_string(index=False))
print()
check("ratings sorted best first", ratings.theta_mean.is_monotonic_increasing)
check("every interval contains its mean", ((ratings.theta_lo < ratings.theta_mean) & (ratings.theta_mean < ratings.theta_hi)).all())
if len(small):
    check("small samples stay near average", (small.theta_mean - 1).abs().max() < 0.15,
          f"{len(small)} goalies with 5 or fewer games")
check("bigger samples have narrower intervals",
      (big.theta_hi - big.theta_lo).mean() < (ratings.theta_hi - ratings.theta_lo).mean())

Best (big samples):
   goalie_name team  games     xga  ga   gsax  theta_mean  theta_lo  theta_hi
   L. Thompson  WSH    101 289.628 245 44.628       0.885     0.809     0.965
A. Vasilevskiy  TBL    121 309.676 268 41.676       0.898     0.823     0.977
 C. Hellebuyck  WPG    120 323.679 287 36.679       0.913     0.838     0.991
    I. Sorokin  NYI    117 340.557 306 34.557       0.922     0.848     0.998
 I. Shesterkin  NYR    112 324.985 293 31.985       0.925     0.849     1.003

Worst (big samples):
  goalie_name team  games     xga  ga    gsax  theta_mean  theta_lo  theta_hi
  K. Lankinen  VAN     98 276.384 292 -15.616       1.041     0.957     1.129
 S. Bobrovsky  FLA    106 259.496 284 -24.504       1.068     0.980     1.159
E. Merzlikins  CBJ     83 236.233 263 -26.767       1.080     0.988     1.174
J. Binnington  STL     98 245.638 274 -28.362       1.082     0.992     1.176
    S. Ersson  PHI     80 193.536 233 -39.464       1.134     1.034     1.239

PASS   ratings sorted

## 7. Do goalie ratings predict the next season?

Rate every goalie as of the first day of the most recent season (from his previous two seasons),
then compare with how he actually did that season (GA ÷ xGA). Only goalies with a rating and a
real workload that season count.

The table compares prior strengths `k`. A tiny `k` is close to the raw record with no shrinkage;
a huge `k` rates everyone as average. The best `k` predicts the next season most accurately
(lowest error). Use it as the default in `goalie.py`.

In [7]:
last_season = gg.season.max()
season_start = pd.to_datetime(gg[gg.season == last_season].game_date).min()
actual = (gg[gg.season == last_season].groupby("goalie_id")
            .agg(games=("game_id", "nunique"), xga=("xga", "sum"), ga=("ga", "sum")))
MIN_GAMES = 20                       # a real workload; lower it if too few goalies qualify
actual = actual[actual.games >= MIN_GAMES]
actual["actual_theta"] = actual.ga / actual.xga

rows = []
for k in [1, 25, 50, 100, 200, 400]:
    pred = goalie_rating_as_of(gg, season_start, k=k).set_index("goalie_id")
    both = actual.join(pred[["theta_mean"]], how="inner")
    err = ((both.theta_mean - both.actual_theta) ** 2 * both.xga).sum() / both.xga.sum()
    rows.append({"k": k, "goalies": len(both), "weighted_mse": err,
                 "correlation": both.theta_mean.corr(both.actual_theta)})

kt = pd.DataFrame(rows)
print(f"Predicting {last_season} from ratings as of {season_start.date()}")
print(kt.round(5).to_string(index=False))
print()
if kt.goalies.max() < 5:
    print(f"Too few goalies to compare ({kt.goalies.max()}). Lower MIN_GAMES and re-run.")
else:
    best_k = kt.loc[kt.weighted_mse.idxmin(), "k"]
    check("shrinkage beats the raw record", kt.set_index("k").weighted_mse.loc[1] > kt.weighted_mse.min(),
          f"best k = {best_k}")
    check("ratings carry signal into the next season", kt.set_index("k").correlation.loc[best_k] > 0)

Predicting 20252026 from ratings as of 2025-10-07
  k  goalies  weighted_mse  correlation
  1       65       0.02912     -0.16324
 25       65       0.01947     -0.06984
 50       65       0.01709     -0.03458
100       65       0.01520     -0.00644
200       65       0.01387      0.01281
400       65       0.01308      0.02483

PASS   shrinkage beats the raw record   (best k = 400)
PASS   ratings carry signal into the next season


**Reading section 7.** Goalie performance is notoriously noisy from season to season, so
the correlation will be modest even for a good rating. The useful output is the best `k`. If it's
far from 100, change `DEFAULT_K` in `src/models/stochastic/goalie.py`. With only one season tested,
treat it as a rough guide; repeating it for each season would make it firmer.

---
If sections 2, 3 and 5 pass, the Phase 3 numbers are correct. Sections 4 and 7 say how useful they are and which settings to keep.